# backprop-pop-outgrad-loop — worked example 2: Diamond DAG: out = z*z accumulates both grad paths into one leaf

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `backprop-pop-outgrad-loop`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper.
    `.grad` accumulates the leaf gradient at the end of the reverse pass."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

When a single leaf feeds **two arguments of the same op** (e.g. `z * z`), the node's `recipe.parents` is `{0: z, 1: z}`. The driver must **accumulate** both back_fn results into `grads[id(z)]` with `+`, never overwrite. By the time `z` is popped, its slot already holds the merged grad, so `.grad` is written exactly once. This is the classic diamond-DAG correctness check.

## Worked solution

We compute `out = z * z` with one leaf `z`. Analytically `d(out)/dz = 2z`, so summing the seed of ones should give `z.grad == 2*z`.

**Step 1 — seed.** `grads = {id(out): ones_like(out)}`.

**Step 2 — pop `out`.** `out` is a `mul` node whose parents dict is `{0: z, 1: z}` — the SAME leaf object at both arg positions. We loop over both entries:
- arg 0: back_fn is `grad_out * y` = `grad_out * z` (the value of arg 1). Accumulate into `grads[id(z)]`.
- arg 1: back_fn is `grad_out * x` = `grad_out * z` (the value of arg 0). Accumulate AGAIN into `grads[id(z)]`.

Because we use `grads[pid] = grads.get(pid, 0) + gp`, the second contribution adds to the first rather than clobbering it. After this step `grads[id(z)] == grad_out*z + grad_out*z == 2*grad_out*z`.

**Step 3 — pop `z`.** `z` is a leaf, so we write the accumulated `2z` into `z.grad`. It is written ONCE — the accumulation happened in the `grads` dict, not via repeated `.grad` writes.

**Why it works.** The accumulate-don't-overwrite rule is exactly what routes both branches of the diamond into the leaf. If we had overwritten, we'd lose one path and get `z.grad == z` instead of `2z`.

In [ ]:
import numpy as np
import torch as t
from einops import rearrange, reduce, repeat
np.random.seed(0); t.manual_seed(0)

class Recipe:
    def __init__(self, func, args, kwargs, parents):
        self.func = func; self.args = args; self.kwargs = kwargs; self.parents = parents

class MiniTensor:
    def __init__(self, array, recipe=None):
        self.array = array; self.recipe = recipe; self.grad = None

def _mul_back0(go, out, x, y):  return go * y
def _mul_back1(go, out, x, y):  return go * x

def backprop(end_node, end_grad, sorted_graph, back_funcs) -> None:
    grads = {id(end_node): end_grad}
    for node in sorted_graph:
        nid = id(node)
        if nid not in grads:
            continue
        grad_out = grads.pop(nid)
        if node.recipe is None:
            node.grad = grad_out if node.grad is None else node.grad + grad_out
            continue
        for argnum, parent in node.recipe.parents.items():
            bf = back_funcs[(node.recipe.func, argnum)]
            gp = bf(grad_out, node.array, *node.recipe.args, **node.recipe.kwargs)
            pid = id(parent)
            grads[pid] = grads.get(pid, 0) + gp

t.manual_seed(0)
z = MiniTensor(t.randn(4))
out_arr = z.array * z.array
out = MiniTensor(out_arr, Recipe('mul', (z.array, z.array), {}, {0: z, 1: z}))
back_funcs = {('mul', 0): _mul_back0, ('mul', 1): _mul_back1}
backprop(out, t.ones_like(out.array), [out, z], back_funcs)
print('z.grad == 2z:', t.allclose(z.grad, 2 * z.array))
print('z.grad:', z.grad)